# 07 — Bandpower Feature Importance and Muscle-Activity Diagnostic

This notebook investigates whether the current stimulus-classification results may be relying on **muscle-related activity**, especially at the **edge electrodes** and near the **start/end of the time window**.

## Motivation

The earlier broadband coefficient analysis suggested that:
- some of the strongest feature importance appeared near the **edge electrodes**
- and some of the strongest temporal importance appeared near the **start/end** of the cropped window

This raises the possibility that part of the classification performance may be driven by subtle **muscle reactions** rather than only by neural signal.

## Goal

To test this idea, this notebook replaces the raw broadband time-series features with **band-limited rectified power features** and examines feature importance separately for:

- **Theta** (4–8 Hz)
- **Alpha** (8–12 Hz)
- **Gamma** (30–45 Hz)

If **gamma-band importance** is concentrated around the **edge electrodes**, that would be a strong indication that the classifier is relying on **muscle activity**.

## Main outputs

This notebook will produce:

1. **Band importance summary**
2. **Time-point importance for each band**
3. **Electrode topoplot for each band**
4. **One report-ready combined figure**

## Important note

These plots are **diagnostic interpretability outputs**, not final benchmark metrics.
They are intended to help determine **what kind of signal** the classifier is using.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import hilbert

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family, SIGNAL_0P1_45
from oddball_eeg.features import flatten_epochs
from oddball_eeg.io import save_dataframe, save_experiment_metadata
from oddball_eeg.loader import load_subjects_grouped, make_subject_summary_table
from oddball_eeg.models import get_model
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)

## 1. Load the experiment configuration

We reuse the same experiment configuration as the main within-subject benchmark so that the analysis stays aligned with the current thesis pipeline.

By default, this notebook uses **all subjects** for the final diagnostic figure.
If you want a faster debugging run first, you can temporarily restrict the subject list to:

- `"006"`
- `"007"`

which would also align with the current focused EEGNet refinement stage.

In [ ]:
set_pandas_display_options()

# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds003061_within_subject_main"
EXPERIMENT_NAME = "ds005863_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)

# Base config is the main within-subject benchmark config.
cfg_base = experiment_family["main"]

# This notebook needs gamma up to 45 Hz, so use a diagnostic 0.1–45 Hz copy.
# This prevents saving gamma diagnostics under the 0.1–12 Hz main benchmark folder.
cfg = replace(
    cfg_base,
    name="bandpower_diagnostic",
    signal=SIGNAL_0P1_45,
)

# Default: use all subjects for a representative diagnostic figure.
# For a faster first pass, replace this with: ["006", "007"]
SUBJECTS_TO_USE = list(cfg.subjects.subjects)

PRIMARY_MODEL = "lda"
BIN_MS = 50

# ERP/P300 interpretation window.
LATE_TMIN = 0.25
LATE_TMAX = 0.50

BANDS_REQUESTED = {
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "gamma": (30.0, 45.0),
}

OUT_TABLE_DIR = cfg.table_dir("interpretability", "bandpower")
OUT_FIG_DIR = cfg.figure_dir("interpretability", "bandpower")

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "07_bandpower_feature_importance.ipynb",
        "purpose": "Bandpower feature-importance and muscle-activity diagnostic.",
        "main_experiment_key": EXPERIMENT_NAME,
        "diagnostic_signal": "0.1-45 Hz",
        "primary_model": PRIMARY_MODEL,
        "subjects_to_use": SUBJECTS_TO_USE,
        "bin_ms": BIN_MS,
        "late_window": [LATE_TMIN, LATE_TMAX],
        "bands_requested": BANDS_REQUESTED,
        "table_dir": str(OUT_TABLE_DIR),
        "figure_dir": str(OUT_FIG_DIR),
    },
)

print("Main experiment key :", EXPERIMENT_NAME)
print("Diagnostic config   :", cfg.name)
print("Dataset             :", cfg.dataset.dataset)
print("Task                :", cfg.dataset.task)
print("Signal config       :", cfg.signal)
print("Subjects            :", SUBJECTS_TO_USE)
print("Primary model       :", PRIMARY_MODEL)
print("Time bin            :", BIN_MS, "ms")
print("Late/ERP window     :", (LATE_TMIN, LATE_TMAX))
print("Table dir           :", OUT_TABLE_DIR)
print("Figure dir          :", OUT_FIG_DIR)

## 2. Load the grouped benchmark dataset and apply the same crop window

We use the same grouped loading logic and the same active crop window as the main benchmark.

This ensures that the bandpower analysis is directly comparable to the current classification setup, while changing only the **feature representation**.

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg)

print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=SUBJECTS_TO_USE,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

crop_window = cfg.effective_crop_window()
assert crop_window is not None, "Expected an active crop window for the main benchmark."

data_main = apply_crop_to_loaded_dataset(
    data_full,
    tmin=crop_window[0],
    tmax=crop_window[1],
)

subject_summary_main = make_subject_summary_table(data_main)
subject_summary_main = round_numeric_columns(subject_summary_main, decimals=4)

display(subject_summary_main)

save_dataframe(
    subject_summary_main,
    OUT_TABLE_DIR / "loaded_subject_summary_main_cropped.csv",
)

## 3. Prepare shared dimensions and adjust the requested bands if needed

The gamma band must remain below the Nyquist frequency.
To make the notebook robust, we automatically clip the upper edge of any band if necessary.

In [ ]:
times = data_main.times
ch_names = data_main.ch_names
sfreq = float(data_main.sfreq)

n_ch = len(ch_names)
n_times = len(times)

nyquist_safe = sfreq / 2.0 - 1.0

bands = {}
for band_name, (l_freq, h_freq) in BANDS_REQUESTED.items():
    h_adj = min(h_freq, nyquist_safe)
    if h_adj <= l_freq:
        raise ValueError(
            f"Band {band_name} invalid after Nyquist adjustment: ({l_freq}, {h_adj})"
        )
    bands[band_name] = (float(l_freq), float(h_adj))

late_mask = (times >= LATE_TMIN) & (times <= LATE_TMAX)
if late_mask.sum() == 0:
    raise ValueError("Late interpretation window is empty after cropping.")

print("Sampling frequency:", sfreq)
print("n_channels        :", n_ch)
print("n_times           :", n_times)
print("Time range        :", float(times[0]), "to", float(times[-1]))
print("Bands used        :", bands)

## 4. Define helper functions for bandpower extraction

For each frequency band, we will:

1. bandpass filter the signal
2. compute the analytic amplitude with the Hilbert transform
3. convert it to rectified power
4. apply a `log1p` transform for stability

This produces a smooth and interpretable bandpower representation for each trial, channel, and time point.

In [ ]:
def bandpass_hilbert_power(
        X: np.ndarray,
        sfreq: float,
        l_freq: float,
        h_freq: float,
) -> np.ndarray:
    """
    Compute log-rectified power for one frequency band.

    Parameters
    ----------
    X : array, shape (n_trials, n_channels, n_times)
    sfreq : float
    l_freq, h_freq : float

    Returns
    -------
    power : array, shape (n_trials, n_channels, n_times)
        log1p(power) after bandpass + Hilbert envelope squared.
    """
    n_trials, n_channels, n_times = X.shape

    # Reshape to (n_signals, n_times) for filtering
    X2 = X.reshape(-1, n_times).astype(np.float64)

    Xf = mne.filter.filter_data(
        X2,
        sfreq=sfreq,
        l_freq=l_freq,
        h_freq=h_freq,
        method="iir",
        verbose=False,
    )

    analytic = hilbert(Xf, axis=-1)
    amp = np.abs(analytic)
    power = amp ** 2
    power = np.log1p(power)

    return power.reshape(n_trials, n_channels, n_times).astype(np.float32)

## 5. Define helper functions for time binning

Bandpower features are often smoother and more interpretable when grouped into short time bins.

Here we use fixed-width bins (default: 50 ms) so that:
- the feature space is smaller
- the coefficient maps are more stable
- the time-importance plots are easier to read

In [ ]:
def bin_time_series(
        X: np.ndarray,
        times: np.ndarray,
        sfreq: float,
        bin_ms: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Bin the time axis into fixed-width bins.

    Parameters
    ----------
    X : array, shape (n_trials, n_channels, n_times)
    times : array, shape (n_times,)
    sfreq : float
    bin_ms : int

    Returns
    -------
    Xb : array, shape (n_trials, n_channels, n_bins)
    bin_centers : array, shape (n_bins,)
    """
    n_trials, n_channels, n_times = X.shape

    bin_samples = max(1, int(round((bin_ms / 1000.0) * sfreq)))
    n_bins = n_times // bin_samples
    if n_bins < 1:
        raise ValueError("Time binning produced zero bins.")

    keep = n_bins * bin_samples
    X_trim = X[..., :keep]
    t_trim = times[:keep]

    Xb = X_trim.reshape(n_trials, n_channels, n_bins, bin_samples).mean(axis=-1)
    tb = t_trim.reshape(n_bins, bin_samples).mean(axis=-1)

    return Xb.astype(np.float32), tb.astype(np.float32)

## 6. Define helper functions for feature assembly and coefficient extraction

We will fit one linear model on the **combined bandpower feature tensor**:

- bands × channels × time bins

The fitted coefficient vector can then be reshaped back into:

- `(n_bands, n_channels, n_bins)`

This makes it possible to inspect:
- importance by band
- importance across time for each band
- importance across electrodes for each band

In [ ]:
def extract_pipeline_coef(clf) -> np.ndarray:
    """
    Extract the coefficient vector from a fitted sklearn pipeline.
    """
    final_est = clf.named_steps["clf"]

    if not hasattr(final_est, "coef_"):
        raise ValueError(f"Final estimator {type(final_est).__name__} has no coef_.")

    coef = np.asarray(final_est.coef_, dtype=np.float64)

    if coef.ndim == 2:
        if coef.shape[0] != 1:
            raise ValueError(f"Expected binary coef shape (1, n_features), got {coef.shape}")
        coef = coef[0]

    return coef.astype(np.float32)


def normalize_importance_map(imp_map: np.ndarray) -> np.ndarray:
    """
    Normalize a subject-level importance map so no subject dominates the aggregate.
    """
    imp = np.abs(imp_map).astype(np.float32)
    denom = float(imp.max()) + 1e-12
    return (imp / denom).astype(np.float32)


def build_band_feature_tensor(
        X: np.ndarray,
        sfreq: float,
        times: np.ndarray,
        bands: dict[str, tuple[float, float]],
        bin_ms: int,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Build a feature tensor of shape:
        (n_trials, n_bands, n_channels, n_bins)
    """
    band_arrays = []
    band_names = list(bands.keys())
    bin_centers = None

    for band_name in band_names:
        l_freq, h_freq = bands[band_name]
        X_power = bandpass_hilbert_power(X, sfreq=sfreq, l_freq=l_freq, h_freq=h_freq)
        X_bin, tb = bin_time_series(X_power, times=times, sfreq=sfreq, bin_ms=bin_ms)

        if bin_centers is None:
            bin_centers = tb
        else:
            if len(tb) != len(bin_centers):
                raise ValueError("Band time-bin lengths do not match.")

        band_arrays.append(X_bin)

    X_stack = np.stack(band_arrays, axis=1)  # (trials, bands, ch, bins)
    return X_stack.astype(np.float32), np.asarray(bin_centers), band_names


def fit_subject_bandpower_model(
        X_band: np.ndarray,
        y: np.ndarray,
        model_name: str,
) -> np.ndarray:
    """
    Fit a linear model on combined bandpower features and return
    coefficient importance of shape (n_bands, n_channels, n_bins).
    """
    n_trials, n_bands, n_ch, n_bins = X_band.shape
    F = X_band.reshape(n_trials, n_bands * n_ch * n_bins)

    clf = get_model(model_name)
    clf.fit(F, y)

    coef_flat = extract_pipeline_coef(clf)
    coef_map = coef_flat.reshape(n_bands, n_ch, n_bins)

    imp_map = normalize_importance_map(coef_map)
    return imp_map.astype(np.float32)

## 7. Define helper functions for topographic plotting

We use an MNE `Info` object with a standard 10–20 montage so we can visualize electrode importance as scalp topoplots.

In [ ]:
def build_mne_info(ch_names: list[str], sfreq: float) -> mne.Info:
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    info.set_montage("standard_1020", on_missing="ignore")
    return info

## 8. Build subject-level bandpower importance summaries

For each subject:
- compute theta, alpha, and gamma bandpower features
- fit the chosen linear model
- reshape the coefficient map into `(band, channel, time-bin)`
- summarize:
  - overall band importance
  - time importance for each band
  - electrode importance for each band

These subject-level summaries will then be averaged across subjects.

In [ ]:
all_rows_band = []
all_rows_time = []
all_rows_elec = []

band_names_ref = None
bin_centers_ref = None

for sub in data_main.subjects:
    X = data_main.X_by_sub[sub]
    y = data_main.y_by_sub[sub]

    print("=" * 70)
    print(f"Subject {sub}")

    X_band, bin_centers, band_names = build_band_feature_tensor(
        X=X,
        sfreq=sfreq,
        times=times,
        bands=bands,
        bin_ms=BIN_MS,
    )

    if band_names_ref is None:
        band_names_ref = band_names
        bin_centers_ref = bin_centers
    else:
        if band_names != band_names_ref:
            raise ValueError("Band names changed across subjects.")
        if len(bin_centers) != len(bin_centers_ref):
            raise ValueError("Bin centers changed across subjects.")

    imp_map = fit_subject_bandpower_model(
        X_band=X_band,
        y=y,
        model_name=PRIMARY_MODEL,
    )  # shape: (bands, ch, bins)

    for b_idx, band_name in enumerate(band_names):
        band_imp = imp_map[b_idx]  # (ch, bins)

        # total importance for the band
        total_band_importance = float(band_imp.mean())

        all_rows_band.append({
            "subject": sub,
            "model": PRIMARY_MODEL,
            "band": band_name,
            "importance_total": total_band_importance,
        })

        # time importance for the band
        time_imp = band_imp.mean(axis=0)
        for t_idx, t in enumerate(bin_centers):
            all_rows_time.append({
                "subject": sub,
                "model": PRIMARY_MODEL,
                "band": band_name,
                "time_s": float(t),
                "importance": float(time_imp[t_idx]),
            })

        # electrode importance for the band
        elec_imp = band_imp.mean(axis=1)
        for ch_idx, ch in enumerate(ch_names):
            all_rows_elec.append({
                "subject": sub,
                "model": PRIMARY_MODEL,
                "band": band_name,
                "channel": ch,
                "importance": float(elec_imp[ch_idx]),
            })

df_band_subject = pd.DataFrame(all_rows_band)
df_time_subject = pd.DataFrame(all_rows_time)
df_elec_subject = pd.DataFrame(all_rows_elec)

save_dataframe(
    df_band_subject,
    OUT_TABLE_DIR / "subject_band_importance.csv",
)

save_dataframe(
    df_time_subject,
    OUT_TABLE_DIR / "subject_band_time_importance.csv",
)

save_dataframe(
    df_elec_subject,
    OUT_TABLE_DIR / "subject_band_electrode_importance.csv",
)

print("Saved subject-level tables:")
print("-", OUT_TABLE_DIR / "subject_band_importance.csv")
print("-", OUT_TABLE_DIR / "subject_band_time_importance.csv")
print("-", OUT_TABLE_DIR / "subject_band_electrode_importance.csv")

## 9. Aggregate the summaries across subjects

We now average the subject-level importance tables to create the report-ready diagnostic outputs.

In [ ]:
df_band_mean = (
    df_band_subject
    .groupby(["model", "band"], as_index=False)
    .agg(
        importance_mean=("importance_total", "mean"),
        importance_std=("importance_total", "std"),
    )
)

df_time_mean = (
    df_time_subject
    .groupby(["model", "band", "time_s"], as_index=False)
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
    )
)

df_elec_mean = (
    df_elec_subject
    .groupby(["model", "band", "channel"], as_index=False)
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
    )
)

save_dataframe(
    df_band_mean,
    OUT_TABLE_DIR / "mean_band_importance.csv",
)

save_dataframe(
    df_time_mean,
    OUT_TABLE_DIR / "mean_band_time_importance.csv",
)

save_dataframe(
    df_elec_mean,
    OUT_TABLE_DIR / "mean_band_electrode_importance.csv",
)

print("Saved aggregated tables:")
print("-", OUT_TABLE_DIR / "mean_band_importance.csv")
print("-", OUT_TABLE_DIR / "mean_band_time_importance.csv")
print("-", OUT_TABLE_DIR / "mean_band_electrode_importance.csv")

display(
    round_numeric_columns(df_band_mean, decimals=4)
)

## 10. Define plotting functions

We create four figure types:

1. **Band importance bar chart**
2. **Time-importance curves by band**
3. **Topoplot for each band**
4. **One combined report-ready figure**

The most important output for the band-specific question is the **band-specific topomap figure**, especially the **gamma** map.

In [ ]:
def plot_band_importance_bar(df_band_mean: pd.DataFrame, model_name: str):
    sub_df = (
        df_band_mean[df_band_mean["model"] == model_name]
        .copy()
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(sub_df["band"], sub_df["importance_mean"])
    ax.set_title(
        f"{model_name.upper()} band importance summary — {cfg.dataset.dataset}, {cfg.dataset.task}"
    )
    ax.set_xlabel("Band")
    ax.set_ylabel("Mean normalized importance")
    ax.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    return fig


def plot_band_time_importance(df_time_mean: pd.DataFrame, model_name: str):
    band_order = ["theta", "alpha", "gamma"]
    fig, axes = plt.subplots(1, len(band_order), figsize=(15, 4), sharey=True)

    for ax, band_name in zip(axes, band_order):
        sub_df = (
            df_time_mean[
                (df_time_mean["model"] == model_name) &
                (df_time_mean["band"] == band_name)
                ]
            .sort_values("time_s")
            .reset_index(drop=True)
        )

        ax.plot(sub_df["time_s"], sub_df["importance_mean"], linewidth=2)
        ax.axvline(0.0, linestyle="--", alpha=0.7)
        ax.axvspan(LATE_TMIN, LATE_TMAX, alpha=0.15)
        ax.set_title(f"{band_name.capitalize()} importance")
        ax.set_xlabel("Time (s)")
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Mean normalized importance")
    fig.suptitle(
        f"{model_name.upper()} time importance by band — {cfg.dataset.dataset}, {cfg.dataset.task}",
        y=1.02,
    )
    fig.tight_layout()
    return fig


def plot_band_topomaps(
        df_elec_mean: pd.DataFrame,
        model_name: str,
        ch_names: list[str],
        sfreq: float,
):
    band_order = ["theta", "alpha", "gamma"]
    info = build_mne_info(ch_names=ch_names, sfreq=sfreq)
    picks = valid_topomap_picks(info)
    info_valid = mne.pick_info(info, picks, copy=True)

    fig, axes = plt.subplots(1, len(band_order), figsize=(15, 4))

    for ax, band_name in zip(axes, band_order):
        sub_df = (
            df_elec_mean[
                (df_elec_mean["model"] == model_name) &
                (df_elec_mean["band"] == band_name)
                ]
            .set_index("channel")
            .reindex(ch_names)
            .reset_index()
        )

        values = sub_df["importance_mean"].to_numpy(dtype=float)
        values_valid = values[picks]

        mne.viz.plot_topomap(
            values_valid,
            info_valid,
            axes=ax,
            show=False,
            contours=6,
            cmap="viridis",
        )
        ax.set_title(f"{band_name.capitalize()}")

    fig.suptitle(
        f"{model_name.upper()} electrode importance by band — {cfg.dataset.dataset}, {cfg.dataset.task}",
        y=1.02,
    )
    fig.tight_layout()
    return fig


def plot_bandpower_report_figure(
        df_time_mean: pd.DataFrame,
        df_elec_mean: pd.DataFrame,
        model_name: str,
        ch_names: list[str],
        sfreq: float,
):
    band_order = ["theta", "alpha", "gamma"]

    info = build_mne_info(ch_names=ch_names, sfreq=sfreq)
    picks = valid_topomap_picks(info)
    info_valid = mne.pick_info(info, picks, copy=True)

    fig, axes = plt.subplots(2, len(band_order), figsize=(15, 8), sharey="row")

    # Top row: time importance
    for j, band_name in enumerate(band_order):
        ax = axes[0, j]
        sub_df = (
            df_time_mean[
                (df_time_mean["model"] == model_name) &
                (df_time_mean["band"] == band_name)
                ]
            .sort_values("time_s")
            .reset_index(drop=True)
        )

        ax.plot(sub_df["time_s"], sub_df["importance_mean"], linewidth=2)
        ax.axvline(0.0, linestyle="--", alpha=0.7)
        ax.axvspan(LATE_TMIN, LATE_TMAX, alpha=0.15)
        ax.set_title(f"{band_name.capitalize()} time importance")
        ax.set_xlabel("Time (s)")
        ax.grid(True, alpha=0.3)

    axes[0, 0].set_ylabel("Mean normalized importance")

    # Bottom row: topomaps
    for j, band_name in enumerate(band_order):
        ax = axes[1, j]
        sub_df = (
            df_elec_mean[
                (df_elec_mean["model"] == model_name) &
                (df_elec_mean["band"] == band_name)
                ]
            .set_index("channel")
            .reindex(ch_names)
            .reset_index()
        )

        values = sub_df["importance_mean"].to_numpy(dtype=float)
        values_valid = values[picks]

        mne.viz.plot_topomap(
            values_valid,
            info_valid,
            axes=ax,
            show=False,
            contours=6,
            cmap="viridis",
        )
        ax.set_title(f"{band_name.capitalize()} topoplot")

    fig.suptitle(
        f"{model_name.upper()} bandpower feature importance: time and topography\n"
        f"{cfg.dataset.dataset}, {cfg.dataset.task}",
        y=1.02,
    )
    fig.tight_layout()
    return fig

## 11. Generate and save the diagnostic figures

This cell creates all the main outputs.

The most useful figures for discussion are likely:
- the **1 × 3 band topomap figure**
or
- the **2 × 3 combined report figure**

In [ ]:
fig_band = plot_band_importance_bar(df_band_mean, PRIMARY_MODEL)
save_figure(
    fig_band,
    OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_importance_bar.png",
)
plt.show()

fig_time = plot_band_time_importance(df_time_mean, PRIMARY_MODEL)
save_figure(
    fig_time,
    OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_time_importance.png",
)
plt.show()

fig_topo = plot_band_topomaps(df_elec_mean, PRIMARY_MODEL, ch_names, sfreq)
save_figure(
    fig_topo,
    OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_topomaps.png",
)
plt.show()

fig_report = plot_bandpower_report_figure(
    df_time_mean=df_time_mean,
    df_elec_mean=df_elec_mean,
    model_name=PRIMARY_MODEL,
    ch_names=ch_names,
    sfreq=sfreq,
)

save_figure(
    fig_report,
    OUT_FIG_DIR / f"{PRIMARY_MODEL}_bandpower_report_figure.png",
)
plt.show()

print("Saved figures:")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_importance_bar.png")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_time_importance.png")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_topomaps.png")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_bandpower_report_figure.png")

## 12. Inspect band importance and gamma top electrodes

This final section makes it easy to inspect:
- whether **gamma** carries unusually large total importance
- and which electrodes contribute most strongly within the gamma band

If the strongest gamma importance is concentrated around edge electrodes, that would support the concern that the classifier is partly relying on muscle-related activity.

In [ ]:
band_importance_table = (
    df_band_mean[df_band_mean["model"] == PRIMARY_MODEL]
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

gamma_top_electrodes = (
    df_elec_mean[
        (df_elec_mean["model"] == PRIMARY_MODEL) &
        (df_elec_mean["band"] == "gamma")
        ]
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("Band importance summary:")
display(round_numeric_columns(band_importance_table.copy(), decimals=4))

print("\nTop gamma electrodes:")
display(round_numeric_columns(gamma_top_electrodes.head(15).copy(), decimals=4))

## 13. Save the key summary tables

These files are useful if you want to:
- cite specific bands in your notes
- inspect the gamma-dominant electrodes
- or decide whether the report text needs to be updated more cautiously

In [ ]:
save_dataframe(
    band_importance_table,
    OUT_TABLE_DIR / f"{PRIMARY_MODEL}_band_importance_summary.csv",
)

save_dataframe(
    gamma_top_electrodes,
    OUT_TABLE_DIR / f"{PRIMARY_MODEL}_gamma_top_electrodes.csv",
)

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "07_bandpower_feature_importance.ipynb",
        "status": "completed",
        "main_experiment_key": EXPERIMENT_NAME,
        "diagnostic_signal": "0.1-45 Hz",
        "primary_model": PRIMARY_MODEL,
        "subjects_to_use": SUBJECTS_TO_USE,
        "bin_ms": BIN_MS,
        "late_window": [LATE_TMIN, LATE_TMAX],
        "bands_used": bands,
        "table_dir": str(OUT_TABLE_DIR),
        "figure_dir": str(OUT_FIG_DIR),
        "n_subjects": int(len(data_main.subjects)),
        "n_channels": int(len(data_main.ch_names)),
        "n_times": int(len(data_main.times)),
    },
)

print("Saved tables:")
print("-", OUT_TABLE_DIR / f"{PRIMARY_MODEL}_band_importance_summary.csv")
print("-", OUT_TABLE_DIR / f"{PRIMARY_MODEL}_gamma_top_electrodes.csv")

print("\nUseful figures:")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_importance_bar.png")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_time_importance.png")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_band_topomaps.png")
print("-", OUT_FIG_DIR / f"{PRIMARY_MODEL}_bandpower_report_figure.png")